In [1]:
import pandas as pd
import numpy as np
import json
import os

from sklearn.preprocessing import LabelEncoder

from transformers import AutoTokenizer

C:\Users\danie\AppData\Local\Temp\ipykernel_10840\1813414503.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Let's define some hyper-parameters
hparams = {
  'input_data': 'C:\\Users\\danie\\Documents\\TFM\\tfm\\data\\intermediate_data\\1_dialogues.csv',
}

## 1) Data loading and processing

In [5]:
df = pd.read_csv(hparams['input_data'])

df.head()

,session_id,turn_index,from,transcript,dialog_acts,audio_file,semantics
0,voip-00d76b791d-20130327_010416,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",NaN,NaN
1,voip-00d76b791d-20130327_010416,1,system,"<sys>Hello , welcome to the Cambridge restaura...",NaN,NaN,NaN
2,voip-00d76b791d-20130327_010416,2,user,<user>expensive restaurant in the south part o...,"[{'slots': [['slot', 'food']], 'act': 'request'}]",pt344x_0000993_0001219.wav,"{'json': [{'slots': [['pricerange', 'expensive..."
3,voip-00d76b791d-20130327_010416,3,system,<sys>What kind of food would you like?,NaN,NaN,NaN
4,voip-00d76b791d-20130327_010416,4,user,<user>any,"[{'slots': [['name', 'the good luck chinese fo...",pt344x_0001649_0001680.wav,"{'json': [{'slots': [['this', 'dontcare']], 'a..."


In [7]:
print(f"{df['session_id'].nunique()} dialogues")
print(f"{len(df)} rows")

3235 dialogues
51002 rows


In [8]:
def eval_if_not_none(x):

  if pd.isna(x):
    return x
  return eval(x)

df['dialog_acts'] = df['dialog_acts'].apply(eval_if_not_none)
df['semantics'] = df['semantics'].apply(eval_if_not_none)

In [9]:
possible_acts = set()
for dialog_act in df['dialog_acts']:

  if type(dialog_act) != list:
    continue

  for d in dialog_act:
    act = d['act']
    possible_acts.add(act)

possible_acts

{'canthelp',
 'canthelp.exception',
 'confirm-domain',
 'expl-conf',
 'impl-conf',
 'inform',
 'offer',
 'repeat',
 'reqmore',
 'request',
 'select',
 'welcomemsg'}

In [10]:
def get_labels(dialog_acts):

  labels = []

  if type(dialog_acts) != list:
    return ""

  for d in dialog_acts:
    act = d['act']
    slots = d['slots']


    if act in ['welcomemsg', 'offer', 'canthelp', 'reqmore',
               'confirm-domain',]:
      labels.append(act)

    elif act in ['inform', 'impl-conf', 'expl-conf']:
      for l in slots:
        labels.append(act + '|' + l[0])
    elif act == 'request':
      for l in slots:
        labels.append(act + '|' + l[1])

  labels = sorted(labels)

  labels = '_'.join(labels)

  return labels

In [11]:
df['label'] = df['dialog_acts'].apply(get_labels)
df.head()

,session_id,turn_index,from,transcript,dialog_acts,audio_file,semantics,label
0,voip-00d76b791d-20130327_010416,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",NaN,NaN,welcomemsg
1,voip-00d76b791d-20130327_010416,1,system,"<sys>Hello , welcome to the Cambridge restaura...",NaN,NaN,NaN,
2,voip-00d76b791d-20130327_010416,2,user,<user>expensive restaurant in the south part o...,"[{'slots': [['slot', 'food']], 'act': 'request'}]",pt344x_0000993_0001219.wav,"{'json': [{'slots': [['pricerange', 'expensive...",request|food
3,voip-00d76b791d-20130327_010416,3,system,<sys>What kind of food would you like?,NaN,NaN,NaN,
4,voip-00d76b791d-20130327_010416,4,user,<user>any,"[{'slots': [['name', 'the good luck chinese fo...",pt344x_0001649_0001680.wav,"{'json': [{'slots': [['this', 'dontcare']], 'a...",inform|area_inform|food_inform|pricerange_offer


In [12]:
def add_db_query_tokens(row):
    if "offer" in row['label']:
        # Concatenate the desired string to transcript
        row['transcript'] = f"{row['transcript']}<API_call><DB_result>"
    elif "canthelp" in row['label']:
        # Concatenate the desired string to transcript
        row['transcript'] = f"{row['transcript']}<API_call><no_DB_result>"
    return row

# Apply the function to each row
df = df.apply(add_db_query_tokens, axis=1)

df.head()

,session_id,turn_index,from,transcript,dialog_acts,audio_file,semantics,label
0,voip-00d76b791d-20130327_010416,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",NaN,NaN,welcomemsg
1,voip-00d76b791d-20130327_010416,1,system,"<sys>Hello , welcome to the Cambridge restaura...",NaN,NaN,NaN,
2,voip-00d76b791d-20130327_010416,2,user,<user>expensive restaurant in the south part o...,"[{'slots': [['slot', 'food']], 'act': 'request'}]",pt344x_0000993_0001219.wav,"{'json': [{'slots': [['pricerange', 'expensive...",request|food
3,voip-00d76b791d-20130327_010416,3,system,<sys>What kind of food would you like?,NaN,NaN,NaN,
4,voip-00d76b791d-20130327_010416,4,user,<user>any<API_call><DB_result>,"[{'slots': [['name', 'the good luck chinese fo...",pt344x_0001649_0001680.wav,"{'json': [{'slots': [['this', 'dontcare']], 'a...",inform|area_inform|food_inform|pricerange_offer


In [13]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json: 100%|██████████| 665/665 [00:00<?, ?B/s] 
C:\Users\danie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\danie\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings

In [14]:
# new tokens
new_tokens = ["<sys>", "<user>" ,"<DA_pred>", "<start>"]

# check if the tokens are already in the vocabulary
new_tokens = set(new_tokens) - set(tokenizer.vocab.keys())

new_tokens

{'<DA_pred>', '<start>', '<sys>', '<user>'}

In [15]:
# modify the tokenizer to take into account the new token
tokenizer.add_tokens(list(new_tokens))

# Add padding token
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

1

In [17]:
def concat_last_9_turns(row):

  session_id = row['session_id']
  turn_index = row['turn_index']

  cond_1 = df['session_id'] == session_id
  cond_2 = df['turn_index'] <= turn_index
  sub_df = df[cond_1 & cond_2].sort_values(by='turn_index', ascending=True)

  sub_df = sub_df.iloc[-18:] #18 we consider both 9 turns of the user and 9 turns of the system

  concat_last_9_turns = sub_df['transcript'].sum()

  return concat_last_9_turns

In [18]:
df['chat_history_last_9'] = df.apply(concat_last_9_turns, axis=1)
df.head()

KeyboardInterrupt: 

In [ ]:
chat_history_last_9_tokenized = tokenizer(list(df['chat_history_last_9'].values), padding='longest')

df['chat_history_last_9_tokenized'] = chat_history_last_9_tokenized['input_ids']
df['attention_mask'] = chat_history_last_9_tokenized['attention_mask']

df.head()

,session_id,turn_index,from,transcript,dialog_acts,label,chat_history_last_9,chat_history_last_9_tokenized,attention_mask
0,voip-59bc8a2167-20130328_115953,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",welcomemsg,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,voip-59bc8a2167-20130328_115953,1,system,"<sys>Hello , welcome to the Cambridge restaura...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,voip-59bc8a2167-20130328_115953,2,user,<user>spanish food,"[{'slots': [['slot', 'pricerange']], 'act': 'r...",request|pricerange,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,voip-59bc8a2167-20130328_115953,3,system,"<sys>Would you like something in the cheap , m...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,voip-59bc8a2167-20130328_115953,4,user,<user>cheap<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [ ]:
def get_speaker_text(row):

  transcript = row['transcript']
  source = row['from']

  len_transcript_tokens = len(tokenizer(transcript)['input_ids'])

  if source == 'user':
    return '<user>' * len_transcript_tokens
  elif source == 'system':
    return '<sys>' * len_transcript_tokens

  raise Exception(f"Unexpected source {source}")

df['speaker_text'] = df.apply(get_speaker_text, axis=1)

df.head()

,session_id,turn_index,from,transcript,dialog_acts,label,chat_history_last_9,chat_history_last_9_tokenized,attention_mask,speaker_text
0,voip-59bc8a2167-20130328_115953,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",welcomemsg,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",<user><user>
1,voip-59bc8a2167-20130328_115953,1,system,"<sys>Hello , welcome to the Cambridge restaura...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...
2,voip-59bc8a2167-20130328_115953,2,user,<user>spanish food,"[{'slots': [['slot', 'pricerange']], 'act': 'r...",request|pricerange,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user>
3,voip-59bc8a2167-20130328_115953,3,system,"<sys>Would you like something in the cheap , m...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...
4,voip-59bc8a2167-20130328_115953,4,user,<user>cheap<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...


In [ ]:
def concat_last_9_turns(row):

  session_id = row['session_id']
  turn_index = row['turn_index']

  cond_1 = df['session_id'] == session_id
  cond_2 = df['turn_index'] <= turn_index
  sub_df = df[cond_1 & cond_2].sort_values(by='turn_index', ascending=True)

  sub_df = sub_df.iloc[-18:] #18 we consider both 9 turns of the user and 9 turns of the system

  concat_last_9_turns = sub_df['speaker_text'].sum() #+ '<DA_pred>'

  return concat_last_9_turns

In [ ]:
df['speaker_text_last_9'] = df.apply(concat_last_9_turns, axis=1)
df.head()

,session_id,turn_index,from,transcript,dialog_acts,label,chat_history_last_9,chat_history_last_9_tokenized,attention_mask,speaker_text,speaker_text_last_9
0,voip-59bc8a2167-20130328_115953,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",welcomemsg,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",<user><user>,<user><user>
1,voip-59bc8a2167-20130328_115953,1,system,"<sys>Hello , welcome to the Cambridge restaura...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...
2,voip-59bc8a2167-20130328_115953,2,user,<user>spanish food,"[{'slots': [['slot', 'pricerange']], 'act': 'r...",request|pricerange,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user>,<user><user><sys><sys><sys><sys><sys><sys><sys...
3,voip-59bc8a2167-20130328_115953,3,system,"<sys>Would you like something in the cheap , m...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...
4,voip-59bc8a2167-20130328_115953,4,user,<user>cheap<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...,<user><user><sys><sys><sys><sys><sys><sys><sys...


In [ ]:
speaker_text_last_9_tokenized = tokenizer(list(df['speaker_text_last_9'].values), padding='longest')

df['speaker_text_last_9_tokenized'] = speaker_text_last_9_tokenized['input_ids']

df.head()

,session_id,turn_index,from,transcript,dialog_acts,label,chat_history_last_9,chat_history_last_9_tokenized,attention_mask,speaker_text,speaker_text_last_9,speaker_text_last_9_tokenized
0,voip-59bc8a2167-20130328_115953,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",welcomemsg,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",<user><user>,<user><user>,"[50259, 50259, 50261, 50261, 50261, 50261, 502..."
1,voip-59bc8a2167-20130328_115953,1,system,"<sys>Hello , welcome to the Cambridge restaura...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
2,voip-59bc8a2167-20130328_115953,2,user,<user>spanish food,"[{'slots': [['slot', 'pricerange']], 'act': 'r...",request|pricerange,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user>,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
3,voip-59bc8a2167-20130328_115953,3,system,"<sys>Would you like something in the cheap , m...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
4,voip-59bc8a2167-20130328_115953,4,user,<user>cheap<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."


In [ ]:
assert (df['chat_history_last_9_tokenized'].apply(len) - df['speaker_text_last_9_tokenized'].apply(len)).sum() == 0

In [ ]:
# Add <DA_pred>
DA_pred_token = tokenizer('<DA_pred>')['input_ids'][0]
sys_token = tokenizer('<sys>')['input_ids'][0]

df['chat_history_last_9_tokenized'].apply(lambda x: x.append(DA_pred_token))
df['speaker_text_last_9_tokenized'].apply(lambda x: x.append(sys_token))
df['attention_mask'].apply(lambda x: x.append(1))

df.head()

,session_id,turn_index,from,transcript,dialog_acts,label,chat_history_last_9,chat_history_last_9_tokenized,attention_mask,speaker_text,speaker_text_last_9,speaker_text_last_9_tokenized
0,voip-59bc8a2167-20130328_115953,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",welcomemsg,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",<user><user>,<user><user>,"[50259, 50259, 50261, 50261, 50261, 50261, 502..."
1,voip-59bc8a2167-20130328_115953,1,system,"<sys>Hello , welcome to the Cambridge restaura...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
2,voip-59bc8a2167-20130328_115953,2,user,<user>spanish food,"[{'slots': [['slot', 'pricerange']], 'act': 'r...",request|pricerange,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user>,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
3,voip-59bc8a2167-20130328_115953,3,system,"<sys>Would you like something in the cheap , m...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
4,voip-59bc8a2167-20130328_115953,4,user,<user>cheap<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."


In [ ]:
df[df['session_id'] == 'voip-59bc8a2167-20130328_115953']

,session_id,turn_index,from,transcript,dialog_acts,label,chat_history_last_9,chat_history_last_9_tokenized,attention_mask,speaker_text,speaker_text_last_9,speaker_text_last_9_tokenized
0,voip-59bc8a2167-20130328_115953,0,user,<user><start>,"[{'slots': [], 'act': 'welcomemsg'}]",welcomemsg,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",<user><user>,<user><user>,"[50259, 50259, 50261, 50261, 50261, 50261, 502..."
1,voip-59bc8a2167-20130328_115953,1,system,"<sys>Hello , welcome to the Cambridge restaura...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
2,voip-59bc8a2167-20130328_115953,2,user,<user>spanish food,"[{'slots': [['slot', 'pricerange']], 'act': 'r...",request|pricerange,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user>,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
3,voip-59bc8a2167-20130328_115953,3,system,"<sys>Would you like something in the cheap , m...",None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys><sys><...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
4,voip-59bc8a2167-20130328_115953,4,user,<user>cheap<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
5,voip-59bc8a2167-20130328_115953,5,system,<sys>pizza express is a great restaurant,None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys>,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
6,voip-59bc8a2167-20130328_115953,6,user,<user>address<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
7,voip-59bc8a2167-20130328_115953,7,system,<sys>pizza express is a great restaurant,None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<sys><sys><sys><sys><sys><sys><sys><sys>,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
8,voip-59bc8a2167-20130328_115953,8,user,<user>address<API_call><DB_result>,"[{'slots': [['name', 'pizza express']], 'act':...",offer,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",<user><user><user><user><user><user><user><use...,<user><user><sys><sys><sys><sys><sys><sys><sys...,"[50259, 50259, 50260, 50260, 50260, 50260, 502..."
9,voip-59bc8a2167-20130328_115953,9,system,<sys>pizza express is a great restaurant,None,,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 706

In [ ]:
# eliminar filas system
df = df[df['from'] == 'user'].reset_index(drop=True)

In [ ]:
# keep only the necessary columns
cols_to_keep = ['session_id', 'turn_index', 'chat_history_last_9',
                'chat_history_last_9_tokenized',
                'speaker_text_last_9_tokenized', 'attention_mask', 'label']
df = df[cols_to_keep]

df.head()

,session_id,turn_index,chat_history_last_9,chat_history_last_9_tokenized,speaker_text_last_9_tokenized,attention_mask,label
0,voip-59bc8a2167-20130328_115953,0,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[50259, 50259, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",welcomemsg
1,voip-59bc8a2167-20130328_115953,2,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",request|pricerange
2,voip-59bc8a2167-20130328_115953,4,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",offer
3,voip-59bc8a2167-20130328_115953,6,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",offer
4,voip-59bc8a2167-20130328_115953,8,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",offer


In [ ]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df['label'])

df.head()

,session_id,turn_index,chat_history_last_9,chat_history_last_9_tokenized,speaker_text_last_9_tokenized,attention_mask,label,label_encoded
0,voip-59bc8a2167-20130328_115953,0,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[50259, 50259, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",welcomemsg,24
1,voip-59bc8a2167-20130328_115953,2,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",request|pricerange,23
2,voip-59bc8a2167-20130328_115953,4,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",offer,19
3,voip-59bc8a2167-20130328_115953,6,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",offer,19
4,voip-59bc8a2167-20130328_115953,8,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",offer,19


In [ ]:
# # One-hot encoding of the labels
# one_hot = pd.get_dummies(df['label'])
# df = df.drop('label', axis=1)
# df = df.join(one_hot)

# df.head()

,session_id,turn_index,chat_history_last_9,chat_history_last_9_tokenized,speaker_text_last_9_tokenized,attention_mask,canthelp,confirm-domain,expl-conf|pricerange,impl-conf|food_impl-conf|pricerange_request|area,...,inform|phone_inform|pricerange_offer,inform|phone_offer,inform|postcode_offer,inform|pricerange_offer,offer,reqmore,request|area,request|food,request|pricerange,welcomemsg
0,voip-59bc8a2167-20130328_115953,0,<user><start>,"[50259, 50258, 50261, 50261, 50261, 50261, 502...","[50259, 50259, 50261, 50261, 50261, 50261, 502...","[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,voip-59bc8a2167-20130328_115953,2,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
2,voip-59bc8a2167-20130328_115953,4,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,voip-59bc8a2167-20130328_115953,6,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,voip-59bc8a2167-20130328_115953,8,"<user><start><sys>Hello , welcome to the Cambr...","[50259, 50258, 50260, 15496, 837, 7062, 284, 2...","[50259, 50259, 50260, 50260, 50260, 50260, 502...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
